In [1]:
import pandas   as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
import os
import unidecode
import geopandas as gpd



# chemin du dossier contenant methods.py
path_methods = r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Notebooks\geocodage\GEOCANCER_impact_baises_CPU\methods"

# ajouter ce chemin dans le path d'importation
sys.path.append(path_methods)

# importer le module
import methods
import certifi
import os
from chunkcsv import process_dataframe_in_chunks2
from methods import normalisation_adresse, normalisation_commune, Needleman_Wunch_update, find_most_common_biaises





## 1- LECTURE DU FICHIER  A GEOCODER

In [2]:
df = pd.read_csv(r"H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\excels\cohorte_finale_avec_adresses.csv", sep=';')
df = df.rename(columns={"adresse_1": "adresse", "code_postal_1": "code_postal", "commune_1": "commune"})

iris = gpd.read_file(r"R:\Direction_Data\0_Projets\Projet_CANCAIR\data\zones_geographiques\iris\CONTOURS-IRIS.shp")
gpd_dept = gpd.read_file(r"R:\Direction_Data\0_Projets\Projet_CANCAIR\data\zones_geographiques\departements\DEPARTEMENT.shp")



## 2- PREPARATION POUR LE GEOCODAGE

In [3]:
# Ensure the adresse_clean column exists and df is populated
if 'adresse_clean' not in df.columns:
	df['adresse_brute'] = df['adresse']
	df['adresse_clean'] = df['adresse']

# Convert to string and fill NaN values
df['adresse_clean'] = df['adresse_clean'].fillna('').astype(str)

# Apply the normalization function
df = normalisation_adresse(df, column='adresse_clean')
df[['adresse_brute', 'adresse_clean']]


# Force le code postal en entier puis en texte propre
df['code_postal'] = df['code_postal'].fillna(0).astype(int).astype(str).str.zfill(5)
df['code_postal'] = df['code_postal'].replace('00000', '')


df = normalisation_commune(df, column='commune')

In [4]:
df

,sexe,age_diagnostic,date_diagnostic,stade,type_histologique,statut_tabagique,paquet_annee,mutation_EGFR,mutation_KRAS,mutation_BRAF,...,adresse,code_postal,commune,categorie_age,exposition_tabagique,mutation_AUTRES,mutation,a_mutation_driver,adresse_brute,adresse_clean
0,feminin,79,2016-09-12,Non disponible,Carcinome epidermoide,fumeur,50.0,NaN,NaN,NaN,...,96 RUE DE LA MAREE\n\n,95320,SAINT LEU LA FORET,70-80,Très forte (>40),negative,NO MUTATION,False,96 RUE DE LA MAREE\n\n,96 RUE DE LA MAREE
1,masculin,47,2016-10-04,Non disponible,Carcinome epidermoide,fumeur,30.0,NaN,NaN,positive,...,LA BUTTE D AMOUR / BATIMENT D1\n4 PLACE ROSA P...,95470,VEMARS,<50,Forte (20-40),positive,BRAF,False,LA BUTTE D AMOUR / BATIMENT D1\n4 PLACE ROSA P...,4 PLACE ROSA PARKS
2,masculin,67,2016-07-21,Non disponible,Carcinome epidermoide,fumeur,40.0,NaN,NaN,NaN,...,11 RUE CLOS DU CHAPITRE\n\n,60300,SENLIS,60-70,Forte (20-40),negative,NO MUTATION,False,11 RUE CLOS DU CHAPITRE\n\n,11 RUE CLOS DU CHAPITRE
3,feminin,57,2015-10-22,Non disponible,Adenocarcinome,fumeur,40.0,NaN,positive,NaN,...,30 RUE MAXIME COURTIS\n\n,89100,SENS,50-60,Forte (20-40),positive,KRAS,False,30 RUE MAXIME COURTIS\n\n,30 RUE MAXIME COURTIS
4,feminin,62,2007-11-15,Non disponible,Carcinome epidermoide,fumeur,44.0,NaN,NaN,NaN,...,12 RUE DE L ARCADE\n\n,94220,CHARENTON LE PONT,60-70,Très forte (>40),negative,NO MUTATION,False,12 RUE DE L ARCADE\n\n,12 RUE DE L ARCADE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3399,feminin,85,2023-02-17,Non disponible,Adenocarcinome,fumeur,40.0,NaN,positive,NaN,...,1 ALLEE LOUIS LE NAIN\nLES HOUTRAIS,92500,RUEIL MALMAISON,>80,Forte (20-40),positive,KRAS,False,1 ALLEE LOUIS LE NAIN\nLES HOUTRAIS,1 ALLEE LOUIS LE NAIN LES HOUTRAIS
3400,masculin,71,2023-04-05,Non disponible,Adenocarcinome,fumeur,40.0,NaN,NaN,NaN,...,1 RUE BEETHOVEN\n\n,75016,PARIS,70-80,Forte (20-40),positive,MET,False,1 RUE BEETHOVEN\n\n,1 RUE BEETHOVEN
3401,feminin,64,2023-03-07,Non disponible,Carcinome indifferencie,fumeur,50.0,NaN,positive,NaN,...,3 RUE A.FRANCE\n\n,92370,CHAVILLE,60-70,Très forte (>40),positive,KRAS,False,3 RUE A.FRANCE\n\n,3 RUE A FRANCE
3402,feminin,67,2018-09-21,Non disponible,Adenocarcinome,fumeur,60.0,NaN,positive,NaN,...,70 BOULEVARD DE STRASBOURG\n\n,94130,NOGENT SUR MARNE,60-70,Très forte (>40),positive,"KRAS,TP53",False,70 BOULEVARD DE STRASBOURG\n\n,70 BOULEVARD DE STRASBOURG


## 3- GEOCODAGE

In [5]:
chunk_size = 4000
chunks_dir_path = "../Data/"
chunks_geo_dir_path = "../Data/"

# Nouvel appel (plus clair et compatible avec requests) :
options_de_requete = {'columns': ['adresse_clean', 'code_postal', 'commune']} # Ce dictionnaire sera utilisé par requests.post(..., data=options_de_requete)

process_dataframe_in_chunks2(df, 'patient' ,chunk_size,chunks_dir_path, chunks_geo_dir_path,
                             options_de_requete)

 Début du géocodage ligne par ligne pour ../Data/patient_chunk_0.csv...
   -> 1000 adresses traitées.
   -> 2000 adresses traitées.
   -> 3000 adresses traitées.
✅ Géocodage de ../Data/patient_chunk_0.csv terminé. Résultats enregistrés dans ../Data/patient_chunk_0_geocoded.csv.


In [6]:
df_geocoded  = pd.read_csv("../Data/patient_chunk_0_geocoded.csv.", sep=';')
df_geocoded




,sexe,age_diagnostic,date_diagnostic,stade,type_histologique,statut_tabagique,paquet_annee,mutation_EGFR,mutation_KRAS,mutation_BRAF,...,mutation,a_mutation_driver,adresse_brute,adresse_clean,match,lon,lat,result_label,score,type
0,feminin,79,2016-09-12,Non disponible,Carcinome epidermoide,fumeur,50.0,NaN,NaN,NaN,...,NO MUTATION,False,96 RUE DE LA MAREE\n\n,96 RUE DE LA MAREE,True,2.240281,49.022942,96 Rue de la Marée 95320 Saint-Leu-la-Forêt,0.968198,housenumber
1,masculin,47,2016-10-04,Non disponible,Carcinome epidermoide,fumeur,30.0,NaN,NaN,positive,...,BRAF,False,LA BUTTE D AMOUR / BATIMENT D1\n4 PLACE ROSA P...,4 PLACE ROSA PARKS,True,2.568545,49.064344,Passerelle Roza Parks 95470 Vémars,0.476708,street
2,masculin,67,2016-07-21,Non disponible,Carcinome epidermoide,fumeur,40.0,NaN,NaN,NaN,...,NO MUTATION,False,11 RUE CLOS DU CHAPITRE\n\n,11 RUE CLOS DU CHAPITRE,True,2.594831,49.206873,11 Rue du Clos du Chapitre 60300 Senlis,0.807185,housenumber
3,feminin,57,2015-10-22,Non disponible,Adenocarcinome,fumeur,40.0,NaN,positive,NaN,...,KRAS,False,30 RUE MAXIME COURTIS\n\n,30 RUE MAXIME COURTIS,True,3.299472,48.201391,30 Rue Maxime Courtis 89100 Sens,0.963126,housenumber
4,feminin,62,2007-11-15,Non disponible,Carcinome epidermoide,fumeur,44.0,NaN,NaN,NaN,...,NO MUTATION,False,12 RUE DE L ARCADE\n\n,12 RUE DE L ARCADE,True,2.401941,48.822991,12 Rue de l'Arcade 94220 Charenton-le-Pont,0.959609,housenumber
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3399,feminin,85,2023-02-17,Non disponible,Adenocarcinome,fumeur,40.0,NaN,positive,NaN,...,KRAS,False,1 ALLEE LOUIS LE NAIN\nLES HOUTRAIS,1 ALLEE LOUIS LE NAIN LES HOUTRAIS,True,2.201929,48.866616,1 Allée Louis Le Nain 92500 Rueil-Malmaison,0.674315,housenumber
3400,masculin,71,2023-04-05,Non disponible,Adenocarcinome,fumeur,40.0,NaN,NaN,NaN,...,MET,False,1 RUE BEETHOVEN\n\n,1 RUE BEETHOVEN,True,2.287671,48.858014,1 Rue Beethoven 75016 Paris,0.970554,housenumber
3401,feminin,64,2023-03-07,Non disponible,Carcinome indifferencie,fumeur,50.0,NaN,positive,NaN,...,KRAS,False,3 RUE A.FRANCE\n\n,3 RUE A FRANCE,True,2.193611,48.816805,3 Rue de la Source 92370 Chaville,0.524218,housenumber
3402,feminin,67,2018-09-21,Non disponible,Adenocarcinome,fumeur,60.0,NaN,positive,NaN,...,"KRAS,TP53",False,70 BOULEVARD DE STRASBOURG\n\n,70 BOULEVARD DE STRASBOURG,True,2.480522,48.839638,70 Boulevard de Strasbourg 94130 Nogent-sur-Marne,0.974917,housenumber


## 4- POST-TRAITEMENT 

In [7]:
df_null = df_geocoded[df_geocoded['lat'].isnull() & df_geocoded['lon'].isnull()]
df_null

,sexe,age_diagnostic,date_diagnostic,stade,type_histologique,statut_tabagique,paquet_annee,mutation_EGFR,mutation_KRAS,mutation_BRAF,...,mutation,a_mutation_driver,adresse_brute,adresse_clean,match,lon,lat,result_label,score,type
167,masculin,54,2014-06-17,Non disponible,Non disponible,fumeur,NaN,NaN,NaN,NaN,...,NO MUTATION,False,MATA UTU\nHAHAKE\n,MATA UTU HAHAKE,False,NaN,NaN,NaN,0.0,NaN
223,masculin,44,2018-03-06,III,Carcinome epidermoide,fumeur,60.0,NaN,NaN,NaN,...,NO MUTATION,False,BONOUMIN VILLAGE\n\n,BONOUMIN VILLAGE,False,NaN,NaN,NaN,0.0,NaN
282,feminin,69,2015-01-20,IV,Adenocarcinome,non fumeur,0.0,positive,NaN,NaN,...,EGFR,True,240 AMBASSADOR RUE BENI GARFAT\n\n,240 AMBASSADOR RUE BENI GARFAT,False,NaN,NaN,NaN,0.0,NaN
363,feminin,66,2015-10-06,III,Adenocarcinome,fumeur,22.0,positive,NaN,NaN,...,EGFR,True,P.O.BOX 09018913\n\n,P O BOX 09018913,False,NaN,NaN,NaN,0.0,NaN
484,feminin,40,2005-12-23,Non disponible,Adenocarcinome,non fumeur,0.0,positive,NaN,NaN,...,EGFR,True,VILLA NÃ¿Â° 8 RESIDENCE AFAK\nSEBALA\nALGERIE,VILLA NA?Adeg 8,False,NaN,NaN,NaN,0.0,NaN
879,masculin,73,2016-08-25,III,Adenocarcinome,fumeur,114.0,NaN,NaN,NaN,...,NO MUTATION,False,RUE MOHAMED EL YAZIDI SECTEUR 13\nBLOC A 20 HA...,RUE MOHAMED EL YAZIDI SECTEUR 13 BLOC A 20 HAY...,False,NaN,NaN,NaN,0.0,NaN
1371,masculin,82,2017-01-10,Non disponible,Adenocarcinome,fumeur,NaN,positive,NaN,NaN,...,EGFR,True,30 RUE GENEVIEVE COUTURIER\n1 RES CASTELLLINA ...,30 RUE GENEVIEVE COUTURIER 1 RES CASTELLLINA PARC,False,NaN,NaN,NaN,0.0,NaN
2244,masculin,59,2019-08-13,Non disponible,Non disponible,fumeur,105.0,NaN,NaN,NaN,...,NO MUTATION,False,DOUAR RIAD\nCR OULED HOUSSOUNE\n,DOUAR RIAD CR OULED HOUSSOUNE,False,NaN,NaN,NaN,0.0,NaN
2516,masculin,65,2020-10-29,Non disponible,Carcinome epidermoide,fumeur,40.0,NaN,NaN,NaN,...,NO MUTATION,False,9 RUE ABDELHAK KADMIRI\nMAARIF EXTENSION,9 RUE ABDELHAK KADMIRI MAARIF EXTENSION,False,NaN,NaN,NaN,0.0,NaN


In [8]:
df_matched =  df_geocoded.dropna(subset=['result_label']).copy()
df_matched = df_matched.rename(columns = {'lon':'x', 'lat':'y'})
df_matched

,sexe,age_diagnostic,date_diagnostic,stade,type_histologique,statut_tabagique,paquet_annee,mutation_EGFR,mutation_KRAS,mutation_BRAF,...,mutation,a_mutation_driver,adresse_brute,adresse_clean,match,x,y,result_label,score,type
0,feminin,79,2016-09-12,Non disponible,Carcinome epidermoide,fumeur,50.0,NaN,NaN,NaN,...,NO MUTATION,False,96 RUE DE LA MAREE\n\n,96 RUE DE LA MAREE,True,2.240281,49.022942,96 Rue de la Marée 95320 Saint-Leu-la-Forêt,0.968198,housenumber
1,masculin,47,2016-10-04,Non disponible,Carcinome epidermoide,fumeur,30.0,NaN,NaN,positive,...,BRAF,False,LA BUTTE D AMOUR / BATIMENT D1\n4 PLACE ROSA P...,4 PLACE ROSA PARKS,True,2.568545,49.064344,Passerelle Roza Parks 95470 Vémars,0.476708,street
2,masculin,67,2016-07-21,Non disponible,Carcinome epidermoide,fumeur,40.0,NaN,NaN,NaN,...,NO MUTATION,False,11 RUE CLOS DU CHAPITRE\n\n,11 RUE CLOS DU CHAPITRE,True,2.594831,49.206873,11 Rue du Clos du Chapitre 60300 Senlis,0.807185,housenumber
3,feminin,57,2015-10-22,Non disponible,Adenocarcinome,fumeur,40.0,NaN,positive,NaN,...,KRAS,False,30 RUE MAXIME COURTIS\n\n,30 RUE MAXIME COURTIS,True,3.299472,48.201391,30 Rue Maxime Courtis 89100 Sens,0.963126,housenumber
4,feminin,62,2007-11-15,Non disponible,Carcinome epidermoide,fumeur,44.0,NaN,NaN,NaN,...,NO MUTATION,False,12 RUE DE L ARCADE\n\n,12 RUE DE L ARCADE,True,2.401941,48.822991,12 Rue de l'Arcade 94220 Charenton-le-Pont,0.959609,housenumber
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3399,feminin,85,2023-02-17,Non disponible,Adenocarcinome,fumeur,40.0,NaN,positive,NaN,...,KRAS,False,1 ALLEE LOUIS LE NAIN\nLES HOUTRAIS,1 ALLEE LOUIS LE NAIN LES HOUTRAIS,True,2.201929,48.866616,1 Allée Louis Le Nain 92500 Rueil-Malmaison,0.674315,housenumber
3400,masculin,71,2023-04-05,Non disponible,Adenocarcinome,fumeur,40.0,NaN,NaN,NaN,...,MET,False,1 RUE BEETHOVEN\n\n,1 RUE BEETHOVEN,True,2.287671,48.858014,1 Rue Beethoven 75016 Paris,0.970554,housenumber
3401,feminin,64,2023-03-07,Non disponible,Carcinome indifferencie,fumeur,50.0,NaN,positive,NaN,...,KRAS,False,3 RUE A.FRANCE\n\n,3 RUE A FRANCE,True,2.193611,48.816805,3 Rue de la Source 92370 Chaville,0.524218,housenumber
3402,feminin,67,2018-09-21,Non disponible,Adenocarcinome,fumeur,60.0,NaN,positive,NaN,...,"KRAS,TP53",False,70 BOULEVARD DE STRASBOURG\n\n,70 BOULEVARD DE STRASBOURG,True,2.480522,48.839638,70 Boulevard de Strasbourg 94130 Nogent-sur-Marne,0.974917,housenumber


In [9]:
#faire une jointure spatiale entre iris et les points géocodés en lambert 93
gdf_points = gpd.GeoDataFrame(df_matched, geometry=gpd.points_from_xy(df_matched.x, df_matched.y), crs="EPSG:4326")
gdf_iris = iris.to_crs("EPSG:2154")


In [10]:
# Créer le GeoDataFrame avec le bon CRS (WGS84 = EPSG:4326)
gdf_points = gpd.GeoDataFrame(
    df_matched, 
    geometry=gpd.points_from_xy(df_matched.x, df_matched.y), 
    crs="EPSG:4326"  # WGS84 au lieu de 2154
)

# PUIS reprojeter en Lambert 93
gdf_points = gdf_points.to_crs("EPSG:2154")

# Vérifier la reprojection
print("Nouvelles bounds des points:")
print(gdf_points.total_bounds)
print("\nExemples de coordonnées reprojetées:")
print(gdf_points.geometry.head())

# Refaire la jointure spatiale
gdf_iris = iris.to_crs("EPSG:2154")
gdf_joined = gpd.sjoin(gdf_points, gdf_iris, how="left", predicate='within')
gdf_joined = gdf_joined.drop(columns=['index_right'])

# Prepare gpd_dept with geometry for spatial join
gpd_dept_reproj = gpd_dept.to_crs("EPSG:2154")
gdf_joined_dept = gpd.sjoin(gdf_joined, gpd_dept_reproj[['CODE_DEPT', 'NOM_DEPT','CODE_REG','NOM_REG', 'geometry']], how="left", predicate='within')
gdf_joined_dept = gdf_joined_dept.drop(columns=['index_right'])


# Vérifier le résultat
print(f"\nNombre de points avec IRIS trouvé : {gdf_joined_dept['CODE_IRIS'].notna().sum()}")
print(f"Nombre de points sans IRIS (NULL) : {gdf_joined_dept['CODE_IRIS'].isna().sum()}")

Nouvelles bounds des points:
[-6425388.35008749   547723.06683958 14447655.6969172  20161466.42245642]

Exemples de coordonnées reprojetées:
0    POINT (644435.403 6880605.405)
1    POINT (668468.921 6885028.743)
2    POINT (670471.248 6900871.055)
3    POINT (722249.565 6789046.349)
4    POINT (656090.288 6858269.478)
Name: geometry, dtype: geometry

Nombre de points avec IRIS trouvé : 3382
Nombre de points sans IRIS (NULL) : 13


In [11]:
gdf_joined_dept

,sexe,age_diagnostic,date_diagnostic,stade,type_histologique,statut_tabagique,paquet_annee,mutation_EGFR,mutation_KRAS,mutation_BRAF,...,INSEE_COM,NOM_COM,IRIS,CODE_IRIS,NOM_IRIS,TYP_IRIS,CODE_DEPT,NOM_DEPT,CODE_REG,NOM_REG
0,feminin,79,2016-09-12,Non disponible,Carcinome epidermoide,fumeur,50.0,NaN,NaN,NaN,...,95563,Saint-Leu-la-Forêt,0102,955630102,La Châtaigneraie,H,95,VAL-D'OISE,11,ILE-DE-FRANCE
1,masculin,47,2016-10-04,Non disponible,Carcinome epidermoide,fumeur,30.0,NaN,NaN,positive,...,95641,Vémars,0000,956410000,Vémars,Z,95,VAL-D'OISE,11,ILE-DE-FRANCE
2,masculin,67,2016-07-21,Non disponible,Carcinome epidermoide,fumeur,40.0,NaN,NaN,NaN,...,60612,Senlis,0701,606120701,Bon Secours,H,60,OISE,32,NORD-PAS-DE-CALAIS-PICARDIE
3,feminin,57,2015-10-22,Non disponible,Adenocarcinome,fumeur,40.0,NaN,positive,NaN,...,89387,Sens,0107,893870107,Nouveaux Quartiers,H,89,YONNE,27,BOURGOGNE-FRANCHE-COMTE
4,feminin,62,2007-11-15,Non disponible,Carcinome epidermoide,fumeur,44.0,NaN,NaN,NaN,...,94018,Charenton-le-Pont,0109,940180109,Bercy,H,94,VAL-DE-MARNE,11,ILE-DE-FRANCE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3399,feminin,85,2023-02-17,Non disponible,Adenocarcinome,fumeur,40.0,NaN,positive,NaN,...,92063,Rueil-Malmaison,0703,920630703,Plateau 3,H,92,HAUTS-DE-SEINE,11,ILE-DE-FRANCE
3400,masculin,71,2023-04-05,Non disponible,Adenocarcinome,fumeur,40.0,NaN,NaN,NaN,...,75116,Paris 16e Arrondissement,6221,751166221,Muette 21,H,75,PARIS,11,ILE-DE-FRANCE
3401,feminin,64,2023-03-07,Non disponible,Carcinome indifferencie,fumeur,50.0,NaN,positive,NaN,...,92022,Chaville,0102,920220102,Iris 0102,H,92,HAUTS-DE-SEINE,11,ILE-DE-FRANCE
3402,feminin,67,2018-09-21,Non disponible,Adenocarcinome,fumeur,60.0,NaN,positive,NaN,...,94052,Nogent-sur-Marne,0111,940520111,Les Hauts de Nogent,H,94,VAL-DE-MARNE,11,ILE-DE-FRANCE


In [12]:
# supprimer les points hors de l'ile de france
gdf_joined_idf = gdf_joined_dept[gdf_joined_dept['CODE_REG'] == '11']
gdf_joined_idf

,sexe,age_diagnostic,date_diagnostic,stade,type_histologique,statut_tabagique,paquet_annee,mutation_EGFR,mutation_KRAS,mutation_BRAF,...,INSEE_COM,NOM_COM,IRIS,CODE_IRIS,NOM_IRIS,TYP_IRIS,CODE_DEPT,NOM_DEPT,CODE_REG,NOM_REG
0,feminin,79,2016-09-12,Non disponible,Carcinome epidermoide,fumeur,50.0,NaN,NaN,NaN,...,95563,Saint-Leu-la-Forêt,0102,955630102,La Châtaigneraie,H,95,VAL-D'OISE,11,ILE-DE-FRANCE
1,masculin,47,2016-10-04,Non disponible,Carcinome epidermoide,fumeur,30.0,NaN,NaN,positive,...,95641,Vémars,0000,956410000,Vémars,Z,95,VAL-D'OISE,11,ILE-DE-FRANCE
4,feminin,62,2007-11-15,Non disponible,Carcinome epidermoide,fumeur,44.0,NaN,NaN,NaN,...,94018,Charenton-le-Pont,0109,940180109,Bercy,H,94,VAL-DE-MARNE,11,ILE-DE-FRANCE
5,masculin,82,2016-12-06,Non disponible,Carcinome epidermoide,fumeur,NaN,NaN,NaN,NaN,...,75120,Paris 20e Arrondissement,7910,751207910,Père Lachaise 10,H,75,PARIS,11,ILE-DE-FRANCE
7,masculin,83,2017-10-16,IV,Adenocarcinome,fumeur,30.0,NaN,NaN,NaN,...,93077,Villemomble,0103,930770103,Marnaudes,H,93,SEINE-SAINT-DENIS,11,ILE-DE-FRANCE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3399,feminin,85,2023-02-17,Non disponible,Adenocarcinome,fumeur,40.0,NaN,positive,NaN,...,92063,Rueil-Malmaison,0703,920630703,Plateau 3,H,92,HAUTS-DE-SEINE,11,ILE-DE-FRANCE
3400,masculin,71,2023-04-05,Non disponible,Adenocarcinome,fumeur,40.0,NaN,NaN,NaN,...,75116,Paris 16e Arrondissement,6221,751166221,Muette 21,H,75,PARIS,11,ILE-DE-FRANCE
3401,feminin,64,2023-03-07,Non disponible,Carcinome indifferencie,fumeur,50.0,NaN,positive,NaN,...,92022,Chaville,0102,920220102,Iris 0102,H,92,HAUTS-DE-SEINE,11,ILE-DE-FRANCE
3402,feminin,67,2018-09-21,Non disponible,Adenocarcinome,fumeur,60.0,NaN,positive,NaN,...,94052,Nogent-sur-Marne,0111,940520111,Les Hauts de Nogent,H,94,VAL-DE-MARNE,11,ILE-DE-FRANCE


In [13]:
df_matched.to_csv("../Data/patients_geocoded_france.csv", sep=';', index=False)

gdf_joined_idf.to_csv("../Data/patients_geocoded_idf.csv", sep=';', index=False)

## 5- ANALYSE DES ECHECS

In [14]:
def analyser_alignement(row):
    S = row['adresse_brute']
    T = row['result_label']

    # S'assurer que les deux chaînes sont valides
    if pd.isna(S) or pd.isna(T):
        return 0, 0, [], []

    # Appeler la fonction d'alignement
    char_score, token_score, unmatched_S, unmatched_T = Needleman_Wunch_update(S, T, return_unmatched=True)
    return char_score, token_score, unmatched_S, unmatched_T

# Créer de nouvelles colonnes avec les résultats de l'alignement
df_matched[['char_score', 'token_score', 'not_matched_brute', 'not_matched_geo']] = \
    df_matched.apply(analyser_alignement, axis=1, result_type='expand')

In [15]:
to_clean = [] 
bruits_a_ajouter = find_most_common_biaises(df_matched, to_clean)
print("\n--- Bruits les plus fréquents à ajouter à normalisation_adresse ---\n")
bruits_a_ajouter.head(20)


--- Bruits les plus fréquents à ajouter à normalisation_adresse ---



H:\PFE Loice\Notebooks\Loice_Canc-air_2025\loice_pneumodetect\Notebooks\geocodage\GEOCANCER_impact_baises_CPU\methods\methods.py:540: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  bruits_fin = bruits_vrais[~bruits_vrais.index.str.contains(pattern, regex=True)]


,count
RUE,193
ALLEE,105
BIS,102
AVENUE,62
TER,18
HALL,15
BOULEVARD,15
CHATEAU,15
PLACE,14
DE,12
